In [1]:
from pathlib import Path
import pandas as pd
import sys
!{sys.executable} -m pip install folium
import folium
from folium.plugins import HeatMap

FILE = "Water Quality.csv"  # Change to full path if needed
OUT = Path("task2_gis_outputs")
OUT.mkdir(exist_ok=True)
if not Path(FILE).exists():
    raise FileNotFoundError(f"Cannot find {FILE!r}; set FILE to the full CSV path.")
df = pd.read_csv(FILE, low_memory=False)
needed = {"MonitoringLocationLatitude", "MonitoringLocationLongitude", "ResultValue"}
if needed - set(df.columns):
    raise ValueError(f"Missing columns: {needed-set(df.columns)}")
for c in ["MonitoringLocationLatitude", "MonitoringLocationLongitude", "ResultValue"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df = df.dropna(subset=["MonitoringLocationLatitude", "MonitoringLocationLongitude", "ResultValue"])
df = df[df.MonitoringLocationLatitude.between(-90, 90) & df.MonitoringLocationLongitude.between(-180, 180)]
if df.empty:
    raise ValueError("No valid coordinate and numeric measurement rows found.")
param = df["CharacteristicName"].dropna().astype(str).value_counts().index[0] if "CharacteristicName" in df else None
if param:
    df = df[df["CharacteristicName"].astype(str) == param].copy()
loc_cols = ["MonitoringLocationLatitude", "MonitoringLocationLongitude"]
for c in ["MonitoringLocationID", "MonitoringLocationName", "MonitoringLocationType"]:
    if c in df: loc_cols.append(c)
agg = df.groupby(loc_cols, dropna=False).agg(median_value=("ResultValue", "median"),
                                              measurements=("ResultValue", "size")).reset_index()
if len(agg) > 5000: agg = agg.sample(5000, random_state=42)
m = folium.Map(location=[agg.MonitoringLocationLatitude.mean(), agg.MonitoringLocationLongitude.mean()],
               zoom_start=5, tiles="CartoDB positron")
layer = folium.FeatureGroup(name=f"Locations — {param or 'ResultValue'}")
for _, r in agg.iterrows():
    name = r.get("MonitoringLocationName", r.get("MonitoringLocationID", "Monitoring location"))
    popup = f"<b>{name}</b><br>Parameter: {param or 'ResultValue'}<br>Median: {r.median_value:.5g}<br>Measurements: {int(r.measurements)}"
    folium.CircleMarker([r.MonitoringLocationLatitude, r.MonitoringLocationLongitude],
                        radius=5, popup=folium.Popup(popup, max_width=300), tooltip=str(name),
                        fill=True, fill_opacity=.75).add_to(layer)
layer.add_to(m)
HeatMap([[r.MonitoringLocationLatitude, r.MonitoringLocationLongitude, 1]
         for r in agg.itertuples()], name="Sampling density", radius=18, blur=15).add_to(m)
folium.LayerControl().add_to(m)
m.save(str(OUT / "water_quality_monitoring_map.html"))
agg.to_csv(OUT / "mapped_location_summary.csv", index=False)
print("Saved", (OUT / "water_quality_monitoring_map.html").resolve())
print("Heat layer represents sampling-location density, not contamination severity.")



   ------------- -------------------------- 1/3 [branca]
   -------------------------- ------------- 2/3 [folium]
   -------------------------- ------------- 2/3 [folium]
   ---------------------------------------- 3/3 [folium]




[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Saved C:\Users\YASH\Desktop\BrainyBeam\task 2 web\task2_gis_outputs\water_quality_monitoring_map.html
Heat layer represents sampling-location density, not contamination severity.


c:\Users\YASH\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\folium\raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore
